# NumPy Basics

A hands-on, runnable reference for NumPy — Python's core library for fast numerical computing with arrays. Run each code cell as you go and tweak values to build intuition.

In [2]:
pip install numpy


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
print(np.__version__)

2.5.1


## 1. NumPy Introduction

- NumPy (**Num**erical **Py**thon) provides the `ndarray` — a fast, fixed-type, multi-dimensional array object.
- NumPy arrays are stored in one contiguous block of memory (unlike Python lists, which store pointers to scattered objects), so operations on them are implemented in C and are **much faster** than equivalent pure-Python loops.
- The universal convention is to `import numpy as np`.

In [3]:
import time

# A quick illustration of why NumPy is fast: element-wise multiply, 1,000,000 items
size = 1_000_000
py_list = list(range(size))
np_array = np.arange(size)

start = time.time()
py_result = [x * 2 for x in py_list]
py_time = time.time() - start

start = time.time()
np_result = np_array * 2
np_time = time.time() - start

print(f"pure Python: {py_time:.4f}s")
print(f"NumPy:       {np_time:.4f}s")
print(f"NumPy was ~{py_time / np_time:.0f}x faster")

pure Python: 0.0343s
NumPy:       0.0043s
NumPy was ~8x faster


## 2. Creating NumPy Arrays

- `np.array()` builds an `ndarray` from a Python list/tuple.
- Arrays have a **dimensionality** (`ndim`): 0-D (scalar), 1-D (vector), 2-D (matrix), or higher ("tensor").
- Helper constructors: `np.zeros`, `np.ones`, `np.empty`, `np.arange`, `np.linspace`, `np.eye`, `np.full`.

In [ ]:
# 0-D array (a scalar wrapped as an array)
a0 = np.array(42)

# 1-D array
a1 = np.array([1, 2, 3, 4, 5])

# 2-D array (matrix)
a2 = np.array([[1, 2, 3], [4, 5, 6]])

# 3-D array
a3 = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

for arr in (a0, a1, a2, a3):
    print(f"ndim={arr.ndim} shape={arr.shape} \n{arr}\n")

# Common constructors
print(np.zeros((2, 3)))        # all zeros
print(np.ones((2, 3)))         # all ones
print(np.full((2, 2), 7))      # filled with a constant
print(np.eye(3))               # identity matrix
print(np.arange(0, 10, 2))     # like range(), but returns an array
print(np.linspace(0, 1, 5))    # 5 evenly spaced values from 0 to 1

# You can force a minimum number of dimensions
d = np.array([1, 2, 3, 4], ndmin=5)
print(d, d.shape)

## 3. NumPy Array Indexing

- 1-D arrays index just like Python lists.
- Multi-dimensional arrays use **comma-separated indices**: `arr[row, col]` instead of `arr[row][col]` (though both work).
- Negative indices count from the end, same as lists.

In [ ]:
arr = np.array([1, 2, 3, 4])
print(arr[0], arr[-1])

arr2d = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]])
print(arr2d[0, 1])       # row 0, col 1 -> 2
print(arr2d[1, -1])      # row 1, last col -> 10

arr3d = np.array([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])
print(arr3d[0, 1, 2])    # 6 - block 0, row 1, col 2

## 4. NumPy Array Slicing

- Same `start:stop:step` syntax as Python lists, but works along **every** axis, separated by commas.
- Slicing an array returns a **view**, not a copy (see the Copy vs View section below) — modifying a slice modifies the original array.

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6, 7])
print(arr[1:5])       # index 1 to 4
print(arr[4:])        # index 4 to end
print(arr[:4])        # start to index 3
print(arr[-3:-1])     # negative slicing
print(arr[1:5:2])     # step

arr2d = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]])
print(arr2d[1, 1:4])      # row 1, columns 1-3
print(arr2d[0:2, 2])      # both rows, column 2
print(arr2d[0:2, 1:4])    # sub-matrix

## 5. NumPy Data Types

- Every array has a single `dtype` shared by **all** its elements (unlike Python lists, which can mix types).
- Common dtypes: `i` (int), `u` (unsigned int), `f` (float), `b` (bool), `U` (unicode string), `c` (complex), `S` (bytes/string).
- Use `.astype()` to convert an array to a new dtype (returns a copy).

In [ ]:
arr = np.array([1, 2, 3, 4])
print(arr.dtype)             # int64 (platform-dependent)

arr_str = np.array(["apple", "banana", "cherry"])
print(arr_str.dtype)          # <U6 - unicode string, max length 6

# Explicit dtype at creation
arr_f = np.array([1, 2, 3], dtype="f4")   # 4-byte float
print(arr_f, arr_f.dtype)

# astype() converts and returns a NEW array
arr_float = np.array([1.5, 2.9, 3.1])
arr_int = arr_float.astype(int)           # truncates, doesn't round
print(arr_int, arr_int.dtype)

arr_bool = np.array([1, 0, -3])
print(arr_bool.astype(bool))              # 0 -> False, anything else -> True

# Trying to build an int array from non-numeric strings raises ValueError
try:
    np.array(["1", "two", "3"], dtype="i")
except ValueError as e:
    print("ValueError:", e)

## 6. NumPy Copy vs View

- `.copy()` creates a **new, independent** array — changes to the copy never affect the original.
- `.view()` (and basic slicing) creates a **new array object that shares the same underlying data** — changes to one show up in the other.
- Every array has a `.base` attribute: `None` for an owner (or a copy), and a reference to the original array for a view.

In [ ]:
arr = np.array([1, 2, 3, 4, 5])

copy_arr = arr.copy()
copy_arr[0] = 99
print("original after copy mutation:", arr)     # unaffected

view_arr = arr.view()
view_arr[0] = 99
print("original after view mutation:", arr)      # changed!

print("copy.base:", copy_arr.base)   # None - owns its data
print("view.base is arr:", view_arr.base is arr)  # True - view shares arr's data

# Slicing also returns a view by default
sl = arr[1:3]
sl[0] = -1
print("original after slice mutation:", arr)

## 7. NumPy Array Shape

`.shape` returns a tuple giving the size of the array along each dimension. The number of elements in the tuple equals `.ndim`.

In [ ]:
arr = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])
print(arr.shape)     # (2, 4) - 2 rows, 4 columns
print(arr.ndim)
print(arr.size)      # total number of elements

d5 = np.array([1, 2, 3, 4], ndmin=5)
print(d5.shape)       # (1, 1, 1, 1, 4)

## 8. NumPy Array Reshape

`.reshape()` returns a new **view** (when possible) with a different shape, without changing the underlying data — the total element count must stay the same. Use `-1` in one dimension to let NumPy infer its size automatically.

In [ ]:
arr = np.arange(1, 13)     # 12 elements
print(arr)

reshaped = arr.reshape(4, 3)
print(reshaped)

reshaped3d = arr.reshape(2, 3, 2)
print(reshaped3d)

# Use -1 to auto-calculate one dimension
auto = arr.reshape(3, -1)
print(auto.shape)   # (3, 4) - NumPy figures out 4 automatically

# Reshaping to an incompatible size raises an error
try:
    arr.reshape(5, 5)
except ValueError as e:
    print("ValueError:", e)

# flatten a multi-dimensional array back to 1-D
print(reshaped.reshape(-1))
print(reshaped.flatten())   # equivalent, always returns a copy

## 9. NumPy Array Iterating

- Iterating a 1-D array yields elements; iterating an n-D array yields its sub-arrays (one fewer dimension) — use nested loops or `np.nditer()` to reach scalar elements directly.
- `np.ndenumerate()` gives both the index and the value, useful for tracking position while iterating.

In [ ]:
arr1d = np.array([1, 2, 3])
for x in arr1d:
    print(x)

arr2d = np.array([[1, 2, 3], [4, 5, 6]])
for row in arr2d:            # yields rows, not scalars
    print(row)

for row in arr2d:
    for x in row:
        print(x)

# np.nditer() flattens iteration down to scalars regardless of dimensionality
for x in np.nditer(arr2d):
    print(x)

# np.ndenumerate() gives (index, value) pairs
for idx, x in np.ndenumerate(arr2d):
    print(idx, x)

## 10. NumPy Joining Arrays

- `np.concatenate()` joins arrays along an existing axis.
- `np.stack()` joins arrays along a **new** axis.
- `np.hstack()` / `np.vstack()` / `np.dstack()` are convenience wrappers for stacking horizontally, vertically, or depth-wise.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print(np.concatenate((a, b)))

a2 = np.array([[1, 2], [3, 4]])
b2 = np.array([[5, 6], [7, 8]])
print(np.concatenate((a2, b2), axis=0))   # stack rows
print(np.concatenate((a2, b2), axis=1))   # stack columns

print(np.stack((a, b), axis=0))    # new axis -> 2x3
print(np.stack((a, b), axis=1))    # new axis -> 3x2

print(np.hstack((a, b)))    # horizontal (along columns for 1-D: just concatenates)
print(np.vstack((a, b)))    # vertical (stacks as new rows)
print(np.dstack((a, b)))    # stacks along a third (depth) axis

## 11. NumPy Splitting Arrays

- `np.array_split()` splits an array into a given number of (possibly unequal) sub-arrays — safer than `np.split()`, which requires an exact even split.
- `np.hsplit()` / `np.vsplit()` split along columns / rows for multi-dimensional arrays.

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6])
print(np.array_split(arr, 3))     # even split
print(np.array_split(arr, 4))     # uneven split - handles the remainder gracefully

# np.split() requires an EXACT even split, or it raises an error
try:
    np.split(arr, 4)
except ValueError as e:
    print("ValueError:", e)

arr2d = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
print(np.array_split(arr2d, 2))          # splits along rows by default (axis=0)
print(np.hsplit(arr2d, 2))                # split along columns

## 12. NumPy Searching Arrays

- `np.where()` returns the **indices** where a condition is true.
- `np.searchsorted()` finds the index where a value *would be inserted* to keep a **sorted** array sorted (binary search, O(log n)).

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 4, 4])
print(np.where(arr == 4))            # indices where value == 4
print(np.where(arr % 2 == 0))         # indices of even numbers

sorted_arr = np.array([6, 7, 8, 9])
print(np.searchsorted(sorted_arr, 7))          # leftmost valid insertion point
print(np.searchsorted(sorted_arr, 7, side="right"))  # rightmost valid insertion point
print(np.searchsorted(sorted_arr, [2, 7, 10]))   # multiple values at once

## 13. NumPy Sorting Arrays

`np.sort()` returns a **new, sorted copy** — it does not sort in place. It works on strings and multi-dimensional arrays too (sorting each row independently for 2-D arrays).

In [ ]:
arr = np.array([3, 1, 5, 2, 4])
print(np.sort(arr))     # new sorted array
print(arr)                # original unchanged

print(np.sort(np.array(["banana", "cherry", "apple"])))

arr2d = np.array([[3, 2, 4], [5, 0, 1]])
print(np.sort(arr2d))     # sorts each row independently

# argsort() returns the INDICES that would sort the array
print(np.argsort(arr))

## 14. NumPy Filtering Arrays

Filtering uses a **boolean index array**: a same-length array of `True`/`False` values selects which elements to keep. Comparisons on arrays (`arr > 5`) directly produce such a boolean array, enabling concise, vectorized filtering.

In [ ]:
arr = np.array([41, 42, 43, 44])

# Manual boolean mask
mask = [True, False, True, False]
print(arr[mask])

# Idiomatic: build the mask directly from a condition (vectorized comparison)
filter_mask = arr > 42
print(filter_mask)
print(arr[filter_mask])

# Common one-liner
print(arr[arr % 2 == 0])

## 15. NumPy Random Numbers

`numpy.random` generates arrays of random numbers, following various distributions, much faster than looping with Python's `random` module. A fixed `seed` makes the sequence reproducible — used here so the notebook's output is deterministic.

In [ ]:
rng = np.random.default_rng(seed=42)   # modern, recommended API

print(rng.integers(0, 100, size=5))            # 5 random ints in [0, 100)
print(rng.random(3))                             # 3 random floats in [0.0, 1.0)
print(rng.choice([3, 5, 7, 9], size=3))          # random picks from an array
print(rng.choice([3, 5, 7, 9], size=3, p=[0.1, 0.1, 0.1, 0.7]))  # weighted picks

# Common distributions
print(rng.normal(loc=0, scale=1, size=5))         # normal/Gaussian distribution
print(rng.integers(1, 7, size=10))                # simulate 10 dice rolls

# Shuffling / permutations
arr = np.array([1, 2, 3, 4, 5])
shuffled = rng.permutation(arr)     # returns a new shuffled array
print(shuffled, arr)                 # original unaffected

rng.shuffle(arr)                     # shuffles arr IN PLACE
print(arr)

## 16. NumPy ufuncs (Universal Functions)

- A **ufunc** operates **element-wise** on arrays, is implemented in compiled C, and supports broadcasting — this is the mechanism behind NumPy's speed and its natural `arr + 1` style syntax.
- Plain Python loops over arrays are slow; ufuncs vectorize the operation instead.
- You can build a custom ufunc from any Python function with `np.frompyfunc()`.

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print(np.add(a, b))         # same as a + b, but explicit
print(type(np.add))          # <class 'numpy.ufunc'>

# Vectorizing a plain Python function
def my_add(x, y):
    return x + y

vectorized_add = np.frompyfunc(my_add, 2, 1)   # 2 inputs, 1 output
print(vectorized_add(a, b))

# Checking whether a function is a ufunc
print(isinstance(np.add, np.ufunc))
print(isinstance(np.concatenate, np.ufunc))    # False - concatenate is not a ufunc

## 17. NumPy Simple Arithmetic

Arithmetic operators (`+ - * / % **`) work element-wise on arrays directly, and correspond to ufuncs (`np.add`, `np.subtract`, ...) under the hood.

In [ ]:
a = np.array([10, 20, 30, 40])
b = np.array([1, 2, 3, 4])

print(a + b, np.add(a, b))
print(a - b, np.subtract(a, b))
print(a * b, np.multiply(a, b))
print(a / b, np.divide(a, b))
print(a % b, np.mod(a, b))
print(a ** b[:1], np.power(a, 2))
print(np.divmod(a, b))          # returns (quotient, remainder) as a tuple of arrays
print(np.absolute(np.array([-1, -2, 3])))

## 18. NumPy Rounding Decimals

- `np.trunc()` / `np.fix()` — truncate toward zero (drop the decimal part).
- `np.around()` — round to a given number of decimals (banker's rounding on ties).
- `np.floor()` — round down; `np.ceil()` — round up.

In [ ]:
arr = np.array([-3.1666, 3.6667])

print(np.trunc(arr))
print(np.fix(arr))
print(np.around(3.1666, 2))
print(np.floor(arr))
print(np.ceil(arr))

## 19. NumPy Logs

Built-in log ufuncs: `np.log2()`, `np.log10()`, `np.log()` (natural log, base e). For an arbitrary base, use `np.frompyfunc()` with `math.log`.

In [ ]:
arr = np.arange(1, 6)

print(np.log2(arr))
print(np.log10(arr))
print(np.log(arr))

# Custom base using frompyfunc + math.log
import math
custom_log = np.frompyfunc(lambda x, base: math.log(x, base), 2, 1)
print(custom_log(100, 10))

## 20. NumPy Summations

- `np.sum()` adds elements — unlike `+`, it can sum across one array or multiple, and along a chosen axis.
- `np.cumsum()` returns the **running/cumulative** sum at each position.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1, 2, 3])

print(np.sum([a, b]))                 # sums everything -> 12
print(np.sum([a, b], axis=1))          # sums each array separately -> [6, 6]
print(np.sum([a, b], axis=0))          # sums element-wise across arrays -> [2, 4, 6]

print(np.cumsum(a))       # [1, 3, 6] - running total

## 21. NumPy Products

- `np.prod()` multiplies all elements together (optionally along an axis, or across multiple arrays).
- `np.cumprod()` returns the running/cumulative product.

In [ ]:
arr = np.array([1, 2, 3, 4])
print(np.prod(arr))              # 1*2*3*4 = 24

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(np.prod([a, b]))            # product of everything
print(np.prod([a, b], axis=1))    # product within each array

print(np.cumprod(arr))            # [1, 2, 6, 24] - running product

## 22. NumPy Differences

`np.diff()` computes the discrete difference between consecutive elements (`arr[i+1] - arr[i]`). Pass `n` to apply it repeatedly (Nth-order differences).

In [ ]:
arr = np.array([10, 15, 25, 5])
print(np.diff(arr))          # [5, 10, -20]
print(np.diff(arr, n=2))     # differences of the differences

## 23. NumPy LCM (Lowest Common Multiple)

`np.lcm.reduce()` finds the LCM across an entire array; `np.lcm()` works on a pair of values/arrays.

In [ ]:
print(np.lcm(4, 6))                          # LCM of a single pair
print(np.lcm.reduce([4, 6, 8]))               # LCM across a whole array
print(np.lcm.reduce(np.arange(1, 11)))         # LCM of 1 through 10

## 24. NumPy GCD (Greatest Common Denominator)

`np.gcd()` for a pair, `np.gcd.reduce()` to find the GCD across an entire array.

In [ ]:
print(np.gcd(12, 18))
print(np.gcd.reduce([12, 18, 30]))

## 25. NumPy Trigonometric Functions

`np.sin()`, `np.cos()`, `np.tan()` (and their inverses `arcsin`/`arccos`/`arctan`) operate element-wise, in **radians**. `np.deg2rad()` / `np.rad2deg()` convert between degrees and radians.

In [ ]:
angles = np.array([0, 30, 45, 60, 90])
radians = np.deg2rad(angles)

print(radians)
print(np.sin(radians))
print(np.cos(radians))
print(np.rad2deg(radians))

# Pythagorean theorem helper
base = np.array([3, 12])
perp = np.array([4, 5])
print(np.hypot(base, perp))     # sqrt(base**2 + perp**2)

## 26. NumPy Hyperbolic Functions

`np.sinh()`, `np.cosh()`, `np.tanh()` and their inverses (`arcsinh`, `arccosh`, `arctanh`) — the hyperbolic analogues of the trig functions.

In [ ]:
arr = np.array([1, 2, 3])
print(np.sinh(arr))
print(np.cosh(arr))
print(np.tanh(arr))
print(np.arcsinh(arr))

## 27. NumPy Set Operations

Treat 1-D arrays as mathematical sets: `np.unique()` de-duplicates; `np.union1d()`, `np.intersect1d()`, `np.setdiff1d()`, `np.setxor1d()` mirror Python's set algebra but return sorted arrays.

In [ ]:
a = np.array([1, 2, 3, 4, 4, 2])
b = np.array([3, 4, 5, 6])

print(np.unique(a))                       # de-duplicated, sorted
print(np.union1d(a, b))                    # everything in either
print(np.intersect1d(a, b, assume_unique=False))  # in both
print(np.setdiff1d(a, b, assume_unique=False))     # in a but not b
print(np.setxor1d(a, b, assume_unique=False))      # in exactly one of a, b

## 28. NumPy Broadcasting

When operating on two arrays of **different shapes**, NumPy "broadcasts" the smaller one across the larger one instead of requiring identical shapes — no explicit loop needed. Broadcasting rules: dimensions are compared from the right; they're compatible if equal, or one of them is 1.

In [ ]:
# Scalar broadcast across every element
arr = np.array([1, 2, 3])
print(arr + 10)

# Row vector broadcast across every row of a matrix
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
row = np.array([10, 20, 30])
print(matrix + row)

# Column vector broadcast across every column
col = np.array([[100], [200], [300]])
print(matrix + col)

# Incompatible shapes raise an error
try:
    np.array([1, 2, 3]) + np.array([1, 2])
except ValueError as e:
    print("ValueError:", e)

## 29. NumPy Linear Algebra

The `numpy.linalg` module and `@`/`np.dot()`/`np.matmul()` cover matrix operations: dot products, matrix multiplication, inversion, determinants, and solving linear systems.

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print(np.dot(a, b))          # dot product
print(a @ b)                  # equivalent, using the @ operator
print(np.matmul(a, b))        # equivalent, explicit matrix multiply

print(np.linalg.det(a))       # determinant
print(np.linalg.inv(a))       # inverse
print(np.transpose(a))        # transpose (swap rows/cols)

# Solve a system of linear equations: 2x + y = 5, x + 3y = 10
coeffs = np.array([[2, 1], [1, 3]])
results = np.array([5, 10])
solution = np.linalg.solve(coeffs, results)
print(solution)   # [x, y]

## 30. NumPy Statistics

Aggregate/statistical functions: `np.mean()`, `np.median()`, `np.std()` (standard deviation), `np.var()` (variance), `np.min()`/`np.max()`, `np.percentile()` — all accept an `axis` parameter for multi-dimensional arrays.

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print("mean:", np.mean(arr))
print("median:", np.median(arr))
print("std:", np.std(arr))
print("var:", np.var(arr))
print("min/max:", np.min(arr), np.max(arr))
print("25th/75th percentile:", np.percentile(arr, 25), np.percentile(arr, 75))

matrix = np.array([[1, 2, 3], [4, 5, 6]])
print(np.mean(matrix, axis=0))    # column-wise means
print(np.mean(matrix, axis=1))    # row-wise means